# Problem 1 — Campus courier tour

This teaching notebook follows the HW3 sample section order and the TSP notebook's AbstractModel / MTZ approach. Work through one section at a time. Only Problem 1 is included.

The formulation belongs in your separate PDF. Keep solver output, routes, and the final explanation in this notebook. This starter has not been solver-executed here; run all cells and save your outputs before submission.

### Load packages

Install Microsoft’s Python and Jupyter extensions in VS Code. With Miniconda installed, run these commands in a terminal (not in a Python cell):

```text
conda create -n aipi530 -c conda-forge python=3.11 pyomo pandas ipykernel glpk
conda activate aipi530
python -m ipykernel install --user --name aipi530 --display-name "Python (aipi530)"
```

Open this notebook in VS Code and select **Python (aipi530)** using Select Kernel. Pyomo builds the model; GLPK solves this mixed-integer linear model. The pricing sample uses IPOPT for a different model type.

Setup references: [Pyomo installation](https://pyomo.readthedocs.io/en/stable/getting_started/installation.html), [VS Code notebooks](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

In [1]:
# load the same packages used in the class notebooks
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

# check that the selected notebook environment can find glpk
solver = SolverFactory('glpk')
print('glpk available:', solver.available(exception_flag=False))

glpk available: True


### Create data

The letter `O` is the dispatch office, not the number zero. Each row is a possible directed trip. Distances are symmetric, but we keep both directions to match the professor's TSP code. Diagonal zeros are data placeholders; self-travel will be excluded from the arc set.

**Checkpoint:** check the O–A, A–B, and B–C distances against the homework table before continuing.

In [2]:
DEPOT = 'O'
locations = ['O', 'A', 'B', 'C']

# enter the distance table in the same row and column order
# all distances are in miles

In [3]:
distance_table = pd.DataFrame(
    [[0, 7, 12, 9],
     [7, 0, 6, 8],
     [12, 6, 0, 5],
     [9, 8, 5, 0]],
    index=locations,
    columns=locations
)

# convert the table to the from/to format used in the tsp example
rows = []
for i in locations:
    for j in locations:
        rows.append({'From': i, 'To': j, 'Distance': distance_table.loc[i, j]})
distance_data = pd.DataFrame(rows)
distance_table

,O,A,B,C
O,0,7,12,9
A,7,0,6,8
B,12,6,0,5
C,9,8,5,0


### Define model

**Sets and parameters:** $V=\{O,A,B,C\}$, $A=\{(i,j)\in V\times V:i\ne j\}$, $n=|V|=4$, and $c_{ij}$ is distance in miles. The set $A$ means arcs; the location named A is one member of $V$.

**Variables:** $x_{ij}=1$ when the courier travels directly from $i$ to $j$, and zero otherwise. The auxiliary integer $u_i$ records visit order: $u_O=0$ and $1\le u_i\le n-1$ for nondepot locations.

An AbstractModel defines the model structure first. The data are attached later by `create_instance`.

In [4]:
# create the model structure before attaching the data
m = AbstractModel()
m.V = Set()
m.A = Set(within=m.V * m.V,
          initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j])
m.c = Param(m.V, m.V)
m.n = Param(initialize=lambda m: len(m.V))

# x chooses the travel arcs and u tracks the visit order
m.x = Var(m.A, domain=Binary)
m.u = Var(m.V, domain=NonNegativeIntegers,
          bounds=lambda m, i: (0, 0) if i == DEPOT else (1, m.n - 1))

**Leave and enter once:**

$\sum_{j\in V:j\ne i}x_{ij}=1\quad\forall i\in V$

$\sum_{i\in V:i\ne j}x_{ij}=1\quad\forall j\in V$

Including O makes the courier leave and return to the dispatch office once.

**Checkpoint:** these equations alone permit O→A→O and B→C→B. Each location has one incoming and one outgoing arc, yet the route is disconnected. This is why we need the next constraint.

In [5]:
# leave each location exactly once, including the office
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) == 1
m.LeaveOnce = Constraint(m.V, rule=Leave_Once_Rule)

# enter each location exactly once, including the office
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) == 1
m.EnterOnce = Constraint(m.V, rule=Enter_Once_Rule)

**MTZ subtour elimination:**

$u_i-u_j+(n-1)x_{ij}\le n-2\quad\forall(i,j)\in A,\ i,j\ne O$

If $x_{ij}=1$, rearranging gives $u_j\ge u_i+1$. Visit order must increase along every selected arc between nondepot locations, which makes a closed loop among those locations impossible. If $x_{ij}=0$, the bounds on u make the inequality automatic.

Skip arcs involving O because the return trip closes the tour; visit order cannot keep increasing back to zero.

**Checkpoint:** with n=4, substitute x=1 and simplify the inequality yourself.

In [6]:
# prevent separate loops among the nondepot locations
def MTZ_Rule(m, i, j):
    if i == DEPOT or j == DEPOT:
        return Constraint.Skip
    return m.u[i] - m.u[j] + (m.n - 1) * m.x[i, j] <= m.n - 2
m.MTZ = Constraint(m.A, rule=MTZ_Rule)

# minimize the sum of the selected travel distances
def Min_Distance_Rule(m):
    return sum(m.c[i, j] * m.x[i, j] for (i, j) in m.A)
m.Min_Distance = Objective(rule=Min_Distance_Rule, sense=minimize)

### Create instance

The outer `{None: ...}` is the default data namespace. The `{None: locations}` supplies an unindexed set. The distance dictionary uses `(from, to)` pairs as its keys. This nesting matches the professor's notebooks.

In [7]:
# attach the problem data to the abstract model
instanceData = {None: {
    'V': {None: locations},
    'c': distance_data.set_index(['From', 'To'])['Distance'].to_dict(),
}}
instance = m.create_instance(instanceData)

### Solve

GLPK handles binary and integer variables in this linear formulation. Only read the solution after confirming optimal termination. If GLPK is missing, install it in the environment selected as the notebook kernel.

In [8]:
# solve the instance and check the termination condition
solver = SolverFactory('glpk')
if not solver.available(exception_flag=False):
    raise RuntimeError('glpk was not found; check the installation and notebook kernel')
solution = solver.solve(instance)
print(solution['Solver'])
assert check_optimal_termination(solution), 'solver did not terminate optimally'
print('optimality certified for this mixed-integer linear model')


- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 5
      Number of created subproblems: 5
  Error rc: 0
  Time: 0.011595010757446289

optimality certified for this mixed-integer linear model


### Print solution

Print all x and u values, the objective, and the ordered route. Testing x > 0.5 allows for small numerical tolerances in binary values. Following the selected arcs gives the travel order; sorting location names would not.

In [9]:
# show every route decision, including unused arcs
arc_output = pd.DataFrame([
    {'From': i, 'To': j, 'x': value(instance.x[i, j])}
    for (i, j) in instance.A
])
print(arc_output.to_string(index=False))

# show the auxiliary order variables as well
order_output = pd.DataFrame([
    {'Location': i, 'u': value(instance.u[i])} for i in instance.V
])
print(order_output.to_string(index=False))

# follow the selected arcs starting at the office
next_location = {}
for (i, j) in instance.A:
    if value(instance.x[i, j]) > 0.5:
        next_location[i] = j

tour = [DEPOT]
for step in range(len(locations)):
    tour.append(next_location[tour[-1]])
    if tour[-1] == DEPOT:
        break

# confirm that the printed tour visits every location and returns home
assert len(tour) == len(locations) + 1
assert tour[-1] == DEPOT
assert set(tour[:-1]) == set(locations)

route_rows = []
for k in range(len(tour) - 1):
    i, j = tour[k], tour[k + 1]
    route_rows.append({'Leg': k + 1, 'From': i, 'To': j,
                       'Miles': value(instance.c[i, j])})
print('ordered route:', ' -> '.join(tour))
print(pd.DataFrame(route_rows).to_string(index=False))
print('total distance:', value(instance.Min_Distance), 'miles')

From To   x
   O  A 1.0
   O  B 0.0
   O  C 0.0
   A  O 0.0
   A  B 1.0
   A  C 0.0
   B  O 0.0
   B  A 0.0
   B  C 1.0
   C  O 1.0
   C  A 0.0
   C  B 0.0
ERROR: evaluating object as numeric value: u[O]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized VarData object u[O]


ValueError: No value for uninitialized VarData object u[O]

### Explanation of results

**Fill this text cell in after running the notebook:**

The courier should follow ____. The total distance is ____ miles. The solver termination condition is ____, so ____.

Every location has exactly one arrival and departure, so all enter/leave equalities hold at equality. The MTZ constraints prevent separate loops. To identify which MTZ inequalities are binding, substitute the printed x and u values and check where the two sides are equal.

**Your checks:**
- add the four route-leg distances by hand and compare with the objective;
- check that O appears only at the beginning and end;
- explain why the two-loop example above is infeasible with MTZ;
- consider whether reversing the tour changes its distance, since distances are symmetric.

Do not describe a time-limited feasible result as certified optimal. The solve cell stops unless the solver reports optimal termination. Save the notebook with outputs after you run it.